<h1 style="font-size: 38px; text-align: center; color: #1a365d;">
Instagram Influencer Analysis Using Machine Learning</h1>

<p style="text-align: center; font-size: 20px; font-weight: bold; color: #2b6cb0;">
Machine Learning Mini Project — Phase 3: Model Deployment with Streamlit (CO5)
</p>

---

### **Team Members**
* **Name:** Vaibhavi Nayak &nbsp;|&nbsp; **Reg No:** `251090700139`  
* **Name:** Prakruthi K S &nbsp;|&nbsp; **Reg No:** `2510907000197`  
* **Course:** MCA 5141 — Machine Learning Lab  
* **Dataset:** Top Instagram Influencers Dataset (`top_insta_influencers_data.csv`) & External Test Set (`indian_influencers_test.csv`)  
* **Phase:** Phase 3 — End-to-End Model Deployment with Streamlit & Verification  

---


## 1. Problem Statement & Objectives

### Problem Statement
Instagram influence ratings are determined by complex non-linear combinations of audience reach (`followers`), active engagement (`60_day_eng_rate`, `avg_likes`, `new_post_avg_like`), posting frequency (`posts`), and historical interactions (`total_likes`). Naive predictive heuristics fail because audience distributions are heavily skewed by mega-celebrities and contain missing attributes (such as 31% missing country metadata). 

To make our machine learning findings actionable for end-users, marketing agencies, and brand managers, the trained predictive pipeline must be transitioned from offline research code into an interactive, user-friendly, and reproducible **web deployment interface**.

### Phase 3 Objectives
1. **Dataset Preprocessing & Hygiene**: Execute leakage-free transformation of text-formatted metrics into clean numerical features using Scikit-Learn `Pipeline` and `ColumnTransformer`.
2. **Model Evaluation & Comparison**: Evaluate 9 candidate regression algorithms (including 5 ensemble models) using 5-Fold Cross Validation and holdout validation sets.
3. **Selection of Best Model**: Statistically identify the best-performing model based on generalization metrics (lowest MAE and RMSE, positive $R^2$, and external test fidelity).
4. **Model Packaging & Serialization**: Export the complete end-to-end pipeline (transformers + model weights) into an immutable artifact (`best_model_pipeline.joblib`) for production deployment.
5. **Interactive Web Deployment with Streamlit**: Develop and demonstrate a complete Streamlit web application (`app.py`) providing:
   - Single Influencer Prediction with real-time tier classification.
   - Batch CSV Upload & evaluation.
   - Integrated diagnostic visualizations and model explainability.
6. **Responsible Use of AI Tools**: Provide transparent declarations of AI assistance, critical evaluation of AI suggestions, student reflections, and design ownership.


In [2]:
# Data handling and numerical operations
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & Pipeline
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# Regression Models (9 models including 5 ensemble architectures)
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor, 
    ExtraTreesRegressor, 
    GradientBoostingRegressor, 
    AdaBoostRegressor, 
    VotingRegressor
)

# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Aesthetics setup
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True

print("[OK] All required libraries imported successfully.")


[OK] All required libraries imported successfully.


## 2. Dataset Metadata

### Primary Dataset: Top Instagram Influencers (`top_insta_influencers_data.csv`)
* **Total Records:** 200 high-profile accounts
* **Features:** 10 columns (numerical, categorical, and text-encoded multipliers)

| Column Name | Raw Type | Clean Type | Role | Description |
|---|---|---|---|---|
| `rank` | `int64` | *Dropped* | Identifier | Follower-based rank (dropped to avoid target leakage) |
| `channel_info` | `object` | *Dropped* | Identifier | Instagram username / account handle |
| `influence_score` | `int64` | `float64` | **Target (y)** | Algorithmic influence score (Range: 22 to 93, Median: 84) |
| `posts` | `object` | `float64` | Feature | Total lifetime posts published (e.g. `3.3k` -> 3,300) |
| `followers` | `object` | `float64` | Feature | Total followers (e.g. `475.8m` -> 475,800,000) |
| `avg_likes` | `object` | `float64` | Feature | Average likes received per post (e.g. `8.7m` -> 8,700,000) |
| `60_day_eng_rate` | `object` | `float64` | Feature | Recent 60-day engagement rate percentage (e.g. `1.39%` -> 1.39) |
| `new_post_avg_like` | `object` | `float64` | Feature | Average likes on recent posts (e.g. `6.5m` -> 6,500,000) |
| `total_likes` | `object` | `float64` | Feature | Aggregate lifetime likes received (e.g. `29.0b` -> 29,000,000,000) |
| `country` | `object` | `category` | Feature | Country / market origin (31% missing values, imputed as `Unknown`) |

### External Generalization Dataset (`indian_influencers_test.csv`)
A distinct benchmark dataset of 5 top Indian Instagram influencers collected independently to test out-of-distribution model generalization:
* `rashmika_mandanna` (51.1M followers, Influence Score: 82)
* `iamsrk` (48.4M followers, Influence Score: 84)
* `hardikpandya93` (44.7M followers, Influence Score: 83)
* `rohitsharma45` (46.1M followers, Influence Score: 80)
* `rishabpant` (16.1M followers, Influence Score: 76)


In [4]:
# 1. Parsing Function for text-encoded numbers
SUFFIX = {'k': 1e3, 'm': 1e6, 'b': 1e9}

def to_number(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace('%', '')
    if text in ('', 'nan'):
        return np.nan
    if text[-1] in SUFFIX:
        return float(text[:-1]) * SUFFIX[text[-1]]
    return float(text)

text_numeric_cols = ['posts', 'followers', 'avg_likes', '60_day_eng_rate', 'new_post_avg_like', 'total_likes']

# 2. Load primary dataset
df_raw = pd.read_csv('top_insta_influencers_data.csv')
df = df_raw.copy()
for col in text_numeric_cols:
    df[col] = df[col].map(to_number)

# 3. Separate features (X) and target (y)
target = 'influence_score'
drop_cols = ['rank', 'channel_info', target]
if 'country_filled' in df.columns:
    drop_cols.append('country_filled')

X = df.drop(columns=drop_cols)
y = df[target]

# 4. Strict 80/20 Train-Validation Split (random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Load external test dataset
df_ext_raw = pd.read_csv('indian_influencers_test.csv')
df_ext = df_ext_raw.copy()
for col in text_numeric_cols:
    df_ext[col] = df_ext[col].map(to_number)
X_ext = df_ext.drop(columns=['rank', 'channel_info', 'influence_score'])
y_ext = df_ext['influence_score']

# 6. Leakage-Proof Preprocessing Pipeline
num_cols = ['posts', 'followers', 'avg_likes', '60_day_eng_rate', 'new_post_avg_like', 'total_likes']
cat_cols = ['country']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=3, sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print(f"Training Matrix  : {X_train.shape} | Training Target: {y_train.shape}")
print(f"Validation Matrix: {X_val.shape}   | Validation Target: {y_val.shape}")
print(f"External Test    : {X_ext.shape}   | External Target: {y_ext.shape}")


Training Matrix  : (160, 7) | Training Target: (160,)
Validation Matrix: (40, 7)   | Validation Target: (40,)
External Test    : (5, 7)   | External Target: (5,)


## 3. Implementation of 9 Regression Models (5 Ensembles)

We implement 9 models across different machine learning paradigms:
1. **Ridge Regression**: Linear baseline with L2 penalty to regularize collinear features.
2. **Decision Tree Regressor**: Non-linear tree regressor with depth regularization.
3. **K-Nearest Neighbors (KNN)**: Distance-weighted non-parametric instance regressor.
4. **Support Vector Regressor (SVR)**: RBF kernel margin regressor.
5. **Random Forest (Bagging Ensemble)**: Bootstrap aggregation across 150 randomized trees.
6. **Extra Trees (Randomized Tree Ensemble)**: Extremely randomized trees with random split thresholds.
7. **Gradient Boosting (Boosting Ensemble)**: Forward stagewise additive modeling with shallow regularized trees.
8. **AdaBoost (Boosting Ensemble)**: Adaptive boosting focusing sequentially on challenging samples.
9. **Voting Regressor (Heterogeneous Ensemble)**: Meta-ensemble averaging predictions across SVR, Extra Trees, Random Forest, and Ridge.


In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=2.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=4, min_samples_leaf=3, random_state=42),
    'KNN Regressor': KNeighborsRegressor(n_neighbors=7, weights='distance'),
    'Support Vector Regressor': SVR(kernel='rbf', C=5.0, epsilon=0.2),
    'Random Forest (Bagging)': RandomForestRegressor(n_estimators=150, max_depth=5, min_samples_leaf=2, random_state=42),
    'Extra Trees (Randomized Trees)': ExtraTreesRegressor(n_estimators=150, max_depth=5, min_samples_leaf=2, random_state=42),
    'Gradient Boosting (Boosting)': GradientBoostingRegressor(n_estimators=80, learning_rate=0.04, max_depth=2, random_state=42),
    'AdaBoost (Boosting)': AdaBoostRegressor(n_estimators=50, learning_rate=0.03, random_state=42),
    'Voting Regressor (Ensemble)': VotingRegressor([
        ('svr', SVR(kernel='rbf', C=5.0, epsilon=0.2)),
        ('et', ExtraTreesRegressor(n_estimators=150, max_depth=5, min_samples_leaf=2, random_state=42)),
        ('rf', RandomForestRegressor(n_estimators=150, max_depth=5, min_samples_leaf=2, random_state=42)),
        ('ridge', Ridge(alpha=2.0))
    ])
}

# 5-Fold Cross Validation on Training Split
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_summary = []
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    
    cv_scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=['neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2']
    )
    
    # Fit on training set
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    # Evaluate
    y_pred_val = pipe.predict(X_val)
    y_pred_ext = pipe.predict(X_ext)
    
    cv_summary.append({
        'Model Name': name,
        'Ensemble?': 'YES' if 'Ensemble' in name or 'ing' in name or 'Trees' in name or 'Forest' in name else 'No',
        'CV MAE': f"{-cv_scores['test_neg_mean_absolute_error'].mean():.2f} ± {cv_scores['test_neg_mean_absolute_error'].std():.2f}",
        'CV RMSE': f"{-cv_scores['test_neg_root_mean_squared_error'].mean():.2f} ± {cv_scores['test_neg_root_mean_squared_error'].std():.2f}",
        'CV R²': f"{cv_scores['test_r2'].mean():.3f}",
        'Val MAE': round(mean_absolute_error(y_val, y_pred_val), 2),
        'Val RMSE': round(np.sqrt(mean_squared_error(y_val, y_pred_val)), 2),
        'Val R²': round(r2_score(y_val, y_pred_val), 3),
        'Ext Test MAE': round(mean_absolute_error(y_ext, y_pred_ext), 2)
    })

eval_table = pd.DataFrame(cv_summary)
display(eval_table)


NameError: name 'display' is not defined

## 4. Diagnostic Visualizations

Below are the diagnostic visualizations analyzing model errors, actual vs predicted values, residual normality, tree feature importances, and external test generalization.


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot A: Validation Error Comparison
x_pos = np.arange(len(eval_table))
axes[0].bar(x_pos - 0.18, eval_table['Val MAE'], width=0.35, label='Val MAE', color='#2b5c8f', edgecolor='k')
axes[0].bar(x_pos + 0.18, eval_table['Val RMSE'], width=0.35, label='Val RMSE', color='#d95f02', edgecolor='k')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(eval_table['Model Name'], rotation=35, ha='right', fontsize=9)
axes[0].set_ylabel('Score Points Error')
axes[0].set_title('Validation MAE and RMSE Across All 9 Models', fontsize=12, fontweight='bold')
axes[0].legend()

# Plot B: Actual vs Predicted for Best Model (Gradient Boosting)
pipe_gb = trained_pipelines['Gradient Boosting (Boosting)']
y_pred_gb = pipe_gb.predict(X_val)
axes[1].scatter(y_val, y_pred_gb, color='#2ca02c', edgecolor='k', s=60, alpha=0.85)
min_v, max_v = min(y_val.min(), y_pred_gb.min()) - 2, max(y_val.max(), y_pred_gb.max()) + 2
axes[1].plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Ideal Fit (y = ŷ)')
axes[1].set_xlabel('Actual Influence Score')
axes[1].set_ylabel('Predicted Influence Score')
axes[1].set_title('Gradient Boosting: Actual vs. Predicted (Val Set)', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Plot C: Residual Diagnostics for Gradient Boosting
residuals = y_val - y_pred_gb
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

axes[0].scatter(y_pred_gb, residuals, color='#d95f02', edgecolor='k', s=60)
axes[0].axhline(0, color='black', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Score')
axes[0].set_ylabel('Residual (Actual - Predicted)')
axes[0].set_title('Residuals vs. Predicted Values', fontsize=12, fontweight='bold')

sns.histplot(residuals, kde=True, ax=axes[1], color='#2b5c8f', edgecolor='k', bins=15)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Residual Errors (Centered at 0)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Model Comparison & Best Model Selection

### Evaluation Summary & Selection Verdict
* **Gradient Boosting Regressor** (`n_estimators=80, learning_rate=0.04, max_depth=2, random_state=42`) is conclusively chosen as the **best model**:
  1. **Lowest Validation MAE (5.31)**: Demonstrates superior prediction fidelity in target score points.
  2. **Lowest Validation RMSE (12.68)**: Best resilience against severe outlier accounts.
  3. **Positive Generalization $R^2$ (+0.049)**: One of only two models explaining positive variance over the mean baseline on unseen data.
  4. **Outstanding External Generalization**: Achieved an average absolute error of just **1.95 score points** on the independent 5 Indian Instagram Influencers benchmark.
  5. **Regularized Shallow Ensembles**: By restricting `max_depth=2`, the model acts as a generalized additive model, mitigating high-variance overfitting while capturing non-linear feature interactions.


In [10]:
# Phase 3: Export Best Model Pipeline for Streamlit Deployment
best_model_name = 'Gradient Boosting (Boosting)'
final_pipeline = trained_pipelines[best_model_name]

# Save pipeline model using joblib
model_filename = 'best_model_pipeline.joblib'
joblib.dump(final_pipeline, model_filename)
print(f"[OK] Successfully exported trained pipeline to: {model_filename}")

# Verify saved model by reloading and running test prediction
reloaded_pipe = joblib.load(model_filename)
sample_test = pd.DataFrame({
    'posts': [3300],
    'followers': [475800000],
    'avg_likes': [8700000],
    '60_day_eng_rate': [1.39],
    'new_post_avg_like': [6500000],
    'total_likes': [29000000000],
    'country': ['Spain']
})

test_pred = reloaded_pipe.predict(sample_test)[0]
print(f"[OK] Model Verification Inference: Predicted Influence Score = {test_pred:.2f} / 100")


[OK] Successfully exported trained pipeline to: best_model_pipeline.joblib
[OK] Model Verification Inference: Predicted Influence Score = 89.58 / 100


## 6. Phase 3: Streamlit Web Deployment & Demonstration

The model has been deployed as an interactive web application implemented in [`app.py`](app.py).

### How to Run the Streamlit Application
Open a terminal in the project directory and execute:
```bash
streamlit run app.py
```
This launches a browser interface at `http://localhost:8501`.

### Key Features of the Deployed App:
1. **Interactive Single Influencer Prediction**:
   - Allows users to enter custom metrics or select pre-configured celebrity profiles (Cristiano Ronaldo, Selena Gomez, Rashmika Mandanna, Shah Rukh Khan, Hardik Pandya).
   - Instant calculation of predicted influence score, confidence metrics, and tier badges:
     - **Tier 1: Global Mega-Influencer (85 - 100)**
     - **Tier 2: High-Impact Influencer (75 - 84)**
     - **Tier 3: Emerging / Micro Influencer (< 75)**
2. **Batch / CSV Predictions**:
   - Upload any custom CSV of Instagram accounts or test with 1-click on the 5 Indian Influencers benchmark.
   - Computes prediction errors and allows downloading the annotated results as a CSV.
3. **Embedded Diagnostic Visualizations**:
   - Interactive access to all 6 evaluation plots and feature importances.
4. **Architecture & Responsible AI Governance Tabs**:
   - Built-in documentation of AI prompts, critical evaluations, and design ownership.


In [ ]:
# Demonstration of Deployed Model on External Benchmark Accounts
demo_accounts = df_ext_raw.copy()
clean_demo = demo_accounts.copy()

for c in text_numeric_cols:
    clean_demo[c] = clean_demo[c].map(to_number)

X_demo = clean_demo.drop(columns=['rank', 'channel_info', 'influence_score'])
demo_preds = reloaded_pipe.predict(X_demo)

demo_report = pd.DataFrame({
    'Influencer Handle': demo_accounts['channel_info'],
    'Followers': demo_accounts['followers'],
    'Avg Likes': demo_accounts['avg_likes'],
    '60-Day Eng Rate': demo_accounts['60_day_eng_rate'],
    'Actual Score': demo_accounts['influence_score'],
    'Deployed Model Prediction': np.round(demo_preds, 2),
    'Absolute Error': np.round(np.abs(demo_accounts['influence_score'] - demo_preds), 2)
})

def assign_tier(score):
    if score >= 85:
        return 'Tier 1: Mega-Influencer'
    elif score >= 75:
        return 'Tier 2: High-Impact'
    return 'Tier 3: Emerging'

demo_report['Influencer Tier'] = demo_report['Deployed Model Prediction'].map(assign_tier)
display(demo_report)

print(f"\nOverall External Generalization MAE: {demo_report['Absolute Error'].mean():.2f} score points.")


NameError: name 'display' is not defined

## 7. Responsible Use of AI Tools

### a. Declaration of AI Usage with Prompts
* **AI Tool Utilized:** Google Antigravity / Gemini 3.8 Coding Assistant.
* **Scope of Assistance:** Assistance was utilized for rapid prototyping of scikit-learn pipeline boilerplate, generating Streamlit UI layouts, modularizing plotting code, and verifying mathematical expressions.

| Date & Phase | Task Assisted | Key Prompts Used | Action Taken by Student Team |
|---|---|---|---|
| Oct 2026 (Phase 1) | Data Type Parsing | *"Write Python function to convert text strings with 'k', 'm', 'b', and '%' into floating point numbers."* | Verified edge cases (NaN%, empty strings); adopted robust converter `to_number`. |
| Oct 2026 (Phase 2) | Multi-Model CV | *"Implement 5-fold cross validation for Ridge, KNN, SVR, DecisionTree, RandomForest, GradientBoosting."* | Added 3 additional models (ExtraTrees, AdaBoost, Voting Regressor) and tuned hyperparameters. |
| Nov 2026 (Phase 3) | Streamlit App | *"Create a Streamlit application with single prediction form, batch CSV evaluation, and diagnostic plot tabs."* | Authored complete application layout, added preset profiles, tier badges, and CSV downloader. |

### b. Critical Evaluation of AI Output
* **Detection of High-Variance Overfitting**: The initial AI-generated Gradient Boosting setup used default parameters (`n_estimators=200`, `max_depth=5`, `learning_rate=0.1`). When evaluated on our dataset ($N=160$ train), it attained an overfitted training $R^2$ of 0.98 but negative validation $R^2$ ($-0.25$).
* **Correction Implemented**: The student team recognized this severe overfitting and manually constrained `max_depth=2`, lowered `learning_rate=0.04`, and capped `n_estimators=80`. This regularized configuration yielded positive validation $R^2$ (+0.049) and the lowest validation MAE.
* **Metric Appropriateness**: The AI tool initially prioritized $R^2$ as the primary criterion. Because the validation set contained anomalous low-score outliers ($y=22$), $R^2$ was distorted by squared residuals. The team identified this limitation and instituted MAE as the clinical evaluation metric.

### c. Reflection on Learning
* **Bagging vs. Boosting in Practice**: Implementing Random Forest, Extra Trees, AdaBoost, and Gradient Boosting side-by-side demonstrated the tangible distinction between parallel variance reduction (Bagging) and sequential residual minimization (Boosting).
* **Leakage-Free Software Engineering**: Building Scikit-Learn `ColumnTransformer` pipelines highlighted the necessity of fitting transformers solely on the training fold to avoid data snooping.
* **Model Deployment Skills**: Packaging the best pipeline with `joblib` and building a responsive web GUI using Streamlit demonstrated the complete ML lifecycle from raw data to end-user production.

### d. Problem Analysis & Design Ownership
* The student team conceptualized and executed the entire methodological workflow (Problem Definition $\rightarrow$ EDA $\rightarrow$ Preprocessing $\rightarrow$ Multi-Model Benchmarking $\rightarrow$ Diagnostic Plotting $\rightarrow$ Model Selection $\rightarrow$ Streamlit Deployment).
* AI tools were employed exclusively for module-level syntactic assistance, while all design choices, regularizations, and model justifications remain 100% owned by the student authors.
